# Advanced Transformations

In this lesson you will learn how to shape the data to perform advanced analysis.

## Learning Objectives
By the end of this lesson, you should be able to:
* Know how to transform data schemas
* Know how to unpack nested structures
* Know how to join two data frames
* Know how to transform data values

🕒 **Estimated Time For Completion:** 2 hours
   

**A gentle note: consider starting your cluster now to avoid waiting for it later.**


## Recap of the EV domain

Recall that we did the following in the previous module :

- We introduced the EV Domain and understood the importance of charge point data.
- We did a quick exercise where we converted timestamp in string format to timestamp format for ease of usage and computations later on.
- We also found out the most recent messages per charge point as an exercise.

Proceeding ahead in the context of our EV domain, lets understand what is a **Charge Data Record** (to be called as **CDR** henceforth). A CDR is an important piece of information required by a **Charge Point Operator (CPOs)** to invoice a customer for the charge dispensed in a transaction. CDR records contain the following information

| CDR field | Description |
| --- | --- |
| total_energy | How much energy was dispensed (StopTransactionRequest.meter_stop - StartTransactionRequest.meter_start) |
| total_time |  How long the transaction was (StopTransactionRequest.timestamp - StartTransactionRequest.timestamp) | 

In this exercise we want to get the total Energy dispensed and the total time taken by vehicle to charge itself.

So lets get started to find out the above two values from the data provided.

We will do this calculation from our OCPP Event data. After the Charge Point has registered itself with the CSMS (Charging Station Management System), it sends information via the OCPP protocol about the Transactions, in the following order:

| OCPP Action | OCPP Message Type | Description | Payload |
| --- | --- | --- | -- |
| StartTransaction | Request | Event sent when a Transaction that has been initiated by the car (or by itself on a scheduled basis). This payload contains the start timestamp (`timestamp`) of the charge and the meter reading (`meter_start`) at the time of the event. This does not contain a transaction ID (but the response back to the Charge Point does). | [example json](https://github.com/data-derp/exercise-ev-databricks/blob/main/sample-data/StartTransactionRequest.json)
| StartTransaction | Response | A response sent back from the Central System to the Charge Point upon receiving a StartTransaction request. This payload contains a transaction ID. | [example json](https://github.com/data-derp/exercise-ev-databricks/blob/main/sample-data/StartTransactionResponse.json)
| MeterValues | Request | Message sent at a set frequency (configured per Charge Point) until the Transaction has ended that samples energy throughput at various outlets. Measurand `Energy.Active.Import.Register` gives a cumulative reading of the charge that has been dispensed for the transaction. Measurand `Power.Active.Import` gives the instantaneous charge at the time of reading. This data contains a transaction ID. | [example json](https://github.com/data-derp/exercise-ev-databricks/blob/main/sample-data/MeterValuesRequest.json) |
| MeterValues | Response | A response sent back from the Central System to the Charge Point upon receiving a MeterValues request. | [example json](https://github.com/data-derp/exercise-ev-databricks/blob/main/sample-data/MeterValuesResponse.json) |
| StopTransaction | Request | Event sent when the car has stopped a Transaction. It contains a transaction ID, the stop timestamp of the charge, and the meter reading (`meter_stop`) at the time of the event. | [example json](https://github.com/data-derp/exercise-ev-databricks/blob/main/sample-data/StopTransactionRequest.json) |
| StopTransaction | Response | A response sent back from the Central System to the Charge Point upon receiving a MeterValues request. | [example json](https://github.com/data-derp/exercise-ev-databricks/blob/main/sample-data/StopTransactionResponse.json) |


**NOTE:** The example payloads are decorated with a few extra fields including the `message_id`, `message_type`, and `charge_point_id` which generally comes from the CSMS. The actual payloads from the Charge Point come in the `body` field encoded as a json string.




## Set up this Notebook
Before we get started, we need to quickly set up this notebook by installing a set of helpers, cleaning up your unique working directory (as to not clash with others working in the same space), and setting some variables. Run the following cells using `Shift + Enter`. 

**NOTE:** that if your cluster shuts down, you will need to re-run the cells in this section.

In [0]:
# And uninstall a few helpers we prepared for you
%pip uninstall -y databricks_helpers exercise_ev_databricks_unit_tests

# now install the databricks helpers 
# %pip install git+https://github.com/data-derp/databricks_helpers.git@sr/dbr_17.3_lts_testing
%pip install git+https://github.com/data-derp/databricks_helpers.git

# # now install and also the databricks test cases
# %pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git@sr/dbr_17.3_lts_testing
%pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git

Ignore the warnings to restart Python, if you get any, in the cell above..

In [0]:
from databricks_helpers.databricks_helpers import DataDerpDatabricksHelpers

exercise_name = "total_charge_dispensed_completed"
helpers = DataDerpDatabricksHelpers(dbutils, exercise_name)

current_user = helpers.current_user()
working_directory = helpers.working_directory()
print(f"Your current working directory is: {working_directory}")

## This function CLEARS your current working directory. Only run this if you want a fresh start or if it is the first time you're doing this exercise.
helpers.clean_working_directory()

## THINKING AHEAD

### The Final Shape of Data
Before we start to ingest our data, it's helpful to know in what direction we're going. A good guideline is to know where you want to be and work backwards to your data sources to identify how we'll need to transform or reshape our data. 

**Target Schema**

<br/>

```
root
 |-- charge_point_id: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- start_timestamp: timestamp (nullable = true)
 |-- stop_timestamp: timestamp (nullable = true)
 |-- total_time: double (nullable = true)
 |-- total_energy: double (nullable = true)
```

#### Reflect
Having looked at the example data in the table above, can you trace how to calculate these fields to the various OCPP events?

## DATA INGESTION

### EXERCISE: Read OCPP Data

In the last connection time exercise of the previous module, we already learned how to read in data (and we did it!). Let's read it in again, but this time, to make it easy, it's already filled out. Run the next two cells below to read in our OCPP data.

In [0]:
url = "https://raw.githubusercontent.com/kelseymok/charge-point-simulator-v1.6/main/out/1679654583.csv"
filepath = helpers.download_to_local_dir(url)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

def create_dataframe(filepath: str) -> DataFrame:
    
    custom_schema = StructType([
        StructField("message_id", StringType(), True),
        StructField("message_type", IntegerType(), True),
        StructField("charge_point_id", StringType(), True),
        StructField("action", StringType(), True),
        StructField("write_timestamp", StringType(), True),
        StructField("body", StringType(), True),
    ])
    
    df = spark.read.format("csv") \
        .option("header", True) \
        .option("delimiter", ",") \
        .option("escape", "\\") \
        .schema(custom_schema) \
        .load(filepath)
    return df
    
df = create_dataframe(filepath)
display(df)


We also have a new dataset to pull in, called "**transactions**". While the `StopTransaction` and `MeterValues` payloads contain a `transaction_id`, the `StartTransaction` payload does not. There is a separate Transactions dataset which maps between the `charge_point_id`, an `id_tag` (an RFID card which is optional to authorize a charge), a timestamp of the start of the transaction, and a transaction_id.


## DATA TRANSFORMATION

### EXERCISE: Return only StopTransaction Requests
Remember that we need to calculate the charging time and the amount of energy released for stopped transactions. Before we do that, we need a data frame that only has StopTransaction data. Use the [filter](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.filter.html) method to return only data where the `action == "StopTransaction"` and `message_type == 2`.

In [0]:
from pyspark.sql.functions import col

def return_stop_transaction_requests(input_df: DataFrame) -> DataFrame:
### Put your code here.
    return input_df.filter((col("action") == "StopTransaction") & (col("message_type") == 2))
    
stop_transaction_request_df = df.transform(return_stop_transaction_requests)
display(stop_transaction_request_df)

#### Unit Test

**NOTE:** Inspect carefully what the unit test actually tests: it creates a DataFrame with mock data and calls only the function that it should be testing. For the purposes of this exercise, this is the only in-line unit test (for demonstrative purposes); the remainder of the tests are hidden as to not spoil the solutions of the exercise itself.

In [0]:
import pandas as pd

def test_return_stop_transaction_unit():
    input_pandas = pd.DataFrame([
        {
            "foo": "30e2ed0c-dd61-4fc1-bcb8-f0a8a0f87c0a",
            "message_type": 2,
            "action": "bar",
        },
        {
            "foo": "4496309f-dfc5-403d-a1c1-54d21b9093c1",
            "message_type": 2,
            "action": "StopTransaction",
        },
        {
            "foo": "bb7b2cd0-f140-4ffe-8280-dc462784303d",
            "message_type": 2,
            "action": "zebra",
        }

    ])

    input_df = spark.createDataFrame(
        input_pandas,
        StructType([
            StructField("foo", StringType()),
            StructField("message_type", IntegerType()),
            StructField("action", StringType()),
        ])
    )

    result = input_df.transform(return_stop_transaction_requests)
    result_count = result.count()
    assert result_count == 1, f"expected 1, but got {result_count}"

    result_actions = [x.action for x in result.collect()]
    expected_actions = ["StopTransaction"]
    assert result_actions == expected_actions, f"expect {expected_actions}, but got {result_actions}"

    result_message_type = [x.message_type for x in result.collect()]
    expected_message_type = [2]
    assert result_message_type == expected_message_type, f"expect {expected_message_type}, but got {result_message_type}"

    print("All tests pass! :)")
    
test_return_stop_transaction_unit()

#### E2E Test
And now the test to ensure that our real data is transformed the way we want.

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_return_stoptransaction_e2e

test_return_stoptransaction_e2e(df.transform(return_stop_transaction_requests), display_f=display)

### EXERCISE: Unpack JSON in StopTransaction
Note in the current schema of the StopTransaction Dataframe we just created, the body is a String (containing JSON):

In [0]:
stop_transaction_request_df.printSchema()

That body contains the majority of the important information, including the `transaction_id`, for which we need as a starting point to fetch all other data related to the transaction. Unfortunately, we can't query this string for `transaction_id`:

In [0]:
from pyspark.sql.functions import col

stop_transaction_request_df.select(col("body.transaction_id"))

What happened here? We can't select `"body.transaction_id"` if it is a `string` type.  But we CAN if it is read in as JSON. 

For reference, there are a [variety of ways of handling JSON in a Dataframe](https://sparkbyexamples.com/pyspark/pyspark-json-functions-with-examples/).


**Your turn:**
Unpack the `body` column (currently a JSON string) for just the `StopTransaction` messages into a new column called `new_body` using the `with_column` and `from_json` functions and the following schema:

<br/>

```
root
 |-- meter_stop: integer (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- reason: string (nullable = true)
 |-- id_tag: string (nullable = true)
 |-- transaction_data: array (nullable = true)
```

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StringType, IntegerType, ArrayType, DoubleType, LongType
def stop_transaction_body_schema():
    return StructType([
### Put your code here.
        StructField("meter_stop", IntegerType(), True),
        StructField("timestamp", StringType(), True),
        StructField("transaction_id", IntegerType(), True),
        StructField("reason", StringType(), True),
        StructField("id_tag", StringType(), True),
        StructField("transaction_data", ArrayType(StringType()), True)
        
    ])
    
def convert_stop_transaction_request_json(input_df: DataFrame) -> DataFrame:
    ### YOUR CODE HERE 
### Put your code here.
    return input_df.withColumn("new_body",from_json(col("body"), stop_transaction_body_schema()))
    ###

display(df.transform(return_stop_transaction_requests).transform(convert_stop_transaction_request_json))



#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_convert_stop_transaction_unit

test_convert_stop_transaction_unit(spark, convert_stop_transaction_request_json)

#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_convert_stop_transaction_json_e2e

test_convert_stop_transaction_json_e2e(df.transform(return_stop_transaction_requests).transform(convert_stop_transaction_request_json), display_f=display)

So, can we now query the JSON?

In [0]:
from pyspark.sql.functions import col

stop_transaction_json_df = df.transform(return_stop_transaction_requests).transform(convert_stop_transaction_request_json)
stop_transaction_json_df.select(col("new_body.transaction_id")).show(5)

In [0]:
stop_transaction_json_df.new_body.transaction_id

Yes, we can !

### EXERCISE: Unpack StartTransaction Response
Using the `transaction_id` in the StopTransaction Requests, we can start to build out our target object by using the `transaction_id` to find the related `StartTransaction` Response, and then the related `StartTransaction` Request. Why the weird route? The `StartTransaction` Response has a `transaction_id` and the `StartTransaction` Request doesn't; but the `StartTransaction` Request has valuable information, namely `meter_start` and a `timestamp` value we can use as our `start_timestamp`. The `StartTransaction` Request and Response both have the same `message_id` by design so we can use that to locate the relevant records. We won't traverse this route until later, but it's important to understand where we need to be.

Very similarly to the previous exercise, we need to unpack the `StartTransaction` Response `body` column from a JSON String to JSON so we can eventually join our data on the `transaction_id` column.

The schema of the resulting JSON should be as follows:

<br/>

```
root
 |-- transaction_id: integer (nullable = true)
 |-- id_tag_info: struct  (nullable = true)
      |-- status: string  (nullable = true)
      |-- parent_id_tag: string  (nullable = true)
      |-- expiry_date: string  (nullable = true)
```

**NOTE:** Though this shows up as an exercise in the notebook, the solutions are already provided to make it easy for you !

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StringType, IntegerType

def start_transaction_response_body_schema():

    id_tag_info_schema = StructType([
        StructField("status", StringType(), True),
        StructField("parent_id_tag", StringType(), True),
        StructField("expiry_date", StringType(), True),
    ])

    schema = StructType([
        StructField("transaction_id", IntegerType(), True),
        StructField("id_tag_info", id_tag_info_schema, True)
    ])


    return schema
    
    
def convert_start_transaction_response_json(input_df: DataFrame) -> DataFrame:
    return input_df.withColumn("new_body",from_json(col("body"), start_transaction_response_body_schema()))


display(df.filter((df.action == "StartTransaction") & (df.message_type == 3)).transform(convert_start_transaction_response_json))



#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_convert_start_transaction_response_json_unit

test_convert_start_transaction_response_json_unit(spark, convert_start_transaction_response_json)

#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_convert_start_transaction_response_json_e2e
    
test_convert_start_transaction_response_json_e2e(
    df.filter((df.action == "StartTransaction") & (df.message_type == 3)).transform(convert_start_transaction_response_json), display_f=display
)

### EXERCISE: Unpack StartTransaction Request
We know we'll need to extract a couple of fields from the `StartTransaction` Request events, namely `meter_start` and `timestamp` (eventually our `start_timestamp` in the target object).

As we have done with the `StopTransaction` Request and the `StartTransaction` Response, unpack the `StartTransaction` Request `body` column (currently a JSON String) into a new JSON column called `new_body` with the following schema:

**Note** Please ensure that the new column is named `new_body` because our tests are expecting that name in their asserts. Is that good naming practice? You tell us!

<br/>

```
root
 |-- connector_id: integer (nullable = true)
 |-- id_tag: string  (nullable = true)
 |-- meter_start: integer  (nullable = true)
 |-- timestamp: string  (nullable = true)
 |-- reservation_id: integer  (nullable = true)
```

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StringType, IntegerType

def start_transaction_request_body_schema():
    schema = None
### Put your code here.
    schema=StructType([
        StructField("connector_id", IntegerType(), True),
        StructField("id_tag", StringType(), True),

        StructField("meter_start", IntegerType(), True),
        StructField("timestamp", StringType(), True),
        
        StructField("reservation_id", IntegerType(), True)

    ])

    return schema
    
    
def convert_start_transaction_request_json(input_df: DataFrame) -> DataFrame:
### Put your code here.
    input_df = input_df.withColumn("new_body",from_json(col("body"), start_transaction_request_body_schema()))
    return input_df

display(df.filter((df.action == "StartTransaction") & (df.message_type == 2)).transform(convert_start_transaction_request_json))



#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_convert_start_transaction_request_unit

test_convert_start_transaction_request_unit(spark, convert_start_transaction_request_json)


#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_convert_start_transaction_request_json_e2e

    
test_convert_start_transaction_request_json_e2e(
    df.filter((df.action == "StartTransaction") & (df.message_type == 2)).transform(
        convert_start_transaction_request_json),
    display_f=display
)

### EXERCISE: Find the matching StartTransaction Requests (Inner Join)

Now that we have unpacked the events for `StartTransaction` Request and `StartTransaction` Response, we can find our matching StartTransaction Request for each StartTransaction Response by executing an **inner** [join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html) between the StartTransaction Response and the StartTransaction Request on the column `message_id`. 

Make sure to return the following columns using the [select](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html) function:
* `charge_point_id`
* `transaction_id`
* `meter_start`
* `start_timestamp` (the `timestamp` column from StartTransaction Request)

**NOTE:** Though this shows up as an exercise in the notebook, the solutions are already provided to make it easy for you !

In [0]:
def join_with_start_transaction_request(input_df: DataFrame, join_df: DataFrame) -> DataFrame:

    return input_df.\
        join(join_df, input_df.message_id == join_df.message_id, "inner").\
        select(
            input_df.charge_point_id.alias("charge_point_id"), 
            input_df.new_body.transaction_id.alias("transaction_id"), 
            join_df.new_body.meter_start.alias("meter_start"), 
            join_df.new_body.timestamp.alias("start_timestamp")
        )
    
start_transaction_response_df = df.filter((df.action == "StartTransaction") & (df.message_type == 3)).transform(convert_start_transaction_response_json)
start_transaction_request_df = df.filter((df.action == "StartTransaction") & (df.message_type == 2)).transform(convert_start_transaction_request_json)

start_transaction_df = start_transaction_response_df.transform(join_with_start_transaction_request, start_transaction_request_df)
display(start_transaction_df)

#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_join_with_start_transaction_request_unit

test_join_with_start_transaction_request_unit(spark, join_with_start_transaction_request)

#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_join_with_start_transaction_request_e2e
    
test_join_with_start_transaction_request_e2e(
    df.filter((df.action == "StartTransaction") & (df.message_type == 3)).transform(convert_start_transaction_response_json). \
    transform(join_with_start_transaction_request, df.filter((df.action == "StartTransaction") & (df.message_type == 2)).transform(convert_start_transaction_request_json)),
    display_f=display
)

### EXERCISE: Join Start and Stop Data
Now that we have our `StartTransaction` events joined together, we can now join our Data Frame with the `StopTransaction` Request data that contains `meter_stop` and `timestamp`.

Executing a [**left join**](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html) between `StopTransaction` Request and the `StartTransaction` DataFrame on the column `new_body.transaction_id`. Make sure to return the following columns using the [select](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html) function:
* `charge_point_id`
* `transaction_id`
* `meter_start`
* `meter_stop`
* `start_timestamp`
* `stop_timestamp` (the `timestamp` column from `StopTransaction` Request)

#### Reflect
Do we join the `StartTransaction` DataFrame to our `StopTransaction` Request data or vice versa?

#### Debug
Does your test fail? Sometimes mocked data in a unit test does not have all of the properties that you are expecting. Try to `printSchema()` and compare the output of the command with the output of the test. See any differences?

In [0]:
def join_stop_with_start(input_df: DataFrame, join_df: DataFrame) -> DataFrame:

### Put your code here.
    input_df=input_df.join(join_df, input_df.new_body.transaction_id == join_df.transaction_id, "left").\
        select(
            join_df.charge_point_id.alias("charge_point_id"), 
            input_df.new_body.transaction_id.alias("transaction_id"), 
                        join_df.meter_start.alias("meter_start"),

            input_df.new_body.meter_stop.alias("meter_stop"), 
                        join_df.start_timestamp.alias("start_timestamp"),

            input_df.new_body.timestamp.alias("stop_timestamp"),
        )
    return input_df

result = df.transform(return_stop_transaction_requests). \
transform(convert_stop_transaction_request_json). \
transform(
    join_stop_with_start, 
    df.filter(
        (df.action == "StartTransaction") & (df.message_type == 3)). \
        transform(convert_start_transaction_response_json).\
        transform(
            join_with_start_transaction_request, 
            df.filter(
                (df.action == "StartTransaction") & (df.message_type == 2)
            ).transform(convert_start_transaction_request_json)))
display(result)

#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_join_stop_with_start_unit

test_join_stop_with_start_unit(spark, join_stop_with_start)

#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_join_stop_with_start_e2e
    
test_join_stop_with_start_e2e(
    df.transform(return_stop_transaction_requests). \
        transform(convert_stop_transaction_request_json). \
        transform(
            join_stop_with_start, 
            df.filter(
                (df.action == "StartTransaction") & (df.message_type == 3)). \
                transform(convert_start_transaction_response_json).\
                transform(
                    join_with_start_transaction_request, 
                    df.filter(
                        (df.action == "StartTransaction") & (df.message_type == 2)
                    ).transform(convert_start_transaction_request_json))),
    display_f=display
)

### EXERCISE: Convert the start_timestamp and stop_timestamp fields to timestamp type

At some point soon, we'll need to calculate the time in hours between the `start_timestamp` and `stop_timestamp` columns. However, note that both columns are of type `string`

<br/>

```
root
 |-- charge_point_id: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- meter_start: integer (nullable = true)
 |-- meter_stop: integer (nullable = true)
 |-- start_timestamp: string (nullable = true)
 |-- stop_timestamp: string (nullable = true)
 ```
 
 In this exercise, we'll convert the `start_timestamp` and `stop_timestamp` columns to a timestamp type.
 
 Target schema:

 <br/>
 
 ```
root
 |-- charge_point_id: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- meter_start: integer (nullable = true)
 |-- meter_stop: integer (nullable = true)
 |-- start_timestamp: timestamp (nullable = true)
 |-- stop_timestamp: timestamp (nullable = true)
 ```

In [0]:
from pyspark.sql.functions import to_timestamp

def convert_start_stop_timestamp_to_timestamp_type(input_df: DataFrame) -> DataFrame:
### Put your code here.
    input_df=input_df.withColumn("start_timestamp", to_timestamp(input_df.start_timestamp)).\
        withColumn("stop_timestamp", to_timestamp(input_df.stop_timestamp))
    return input_df

result = df.transform(return_stop_transaction_requests). \
    transform(convert_stop_transaction_request_json). \
    transform(
        join_stop_with_start, 
        df.filter(
            (df.action == "StartTransaction") & (df.message_type == 3)). \
            transform(convert_start_transaction_response_json).\
            transform(
                join_with_start_transaction_request, 
                df.filter(
                    (df.action == "StartTransaction") & (df.message_type == 2)
                ).transform(convert_start_transaction_request_json))). \
    transform(convert_start_stop_timestamp_to_timestamp_type)
display(result)

#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_convert_start_stop_timestamp_to_timestamp_type_unit

test_convert_start_stop_timestamp_to_timestamp_type_unit(spark, convert_start_stop_timestamp_to_timestamp_type)


#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_convert_start_stop_timestamp_to_timestamp_type_e2e

    
test_convert_start_stop_timestamp_to_timestamp_type_e2e(
    df.transform(return_stop_transaction_requests). \
    transform(convert_stop_transaction_request_json). \
    transform(
        join_stop_with_start, 
        df.filter(
            (df.action == "StartTransaction") & (df.message_type == 3)). \
            transform(convert_start_transaction_response_json).\
            transform(
                join_with_start_transaction_request, 
                df.filter(
                    (df.action == "StartTransaction") & (df.message_type == 2)
                ).transform(convert_start_transaction_request_json))). \
    transform(convert_start_stop_timestamp_to_timestamp_type),
    display_f=display
)

### EXERCISE: Calculate the Charge Transaction Duration (total_time)
Now that we have our `start_timestamp` and `stop_timestamp` columns in the appropriate type, we now can calculate the time in hours of the charge by subtracting the `start_timestamp` from the `stop_timestamp` column and doing some arithmetic.

Current schema:

<br/>

```
root
 |-- charge_point_id: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- meter_start: integer (nullable = true)
 |-- meter_stop: integer (nullable = true)
 |-- start_timestamp: timestamp (nullable = true)
 |-- stop_timestamp: timestamp (nullable = true)
```

Expected schema:

<br/>

```
root
 |-- charge_point_id: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- meter_start: integer (nullable = true)
 |-- meter_stop: integer (nullable = true)
 |-- start_timestamp: timestamp (nullable = true)
 |-- stop_timestamp: timestamp (nullable = true)
 |-- total_time: double (nullable = true) 
```

Round to two decimal places (**NOTE:** it might not appear as two decimal places in the resulting Dataframe as a result of rendering).

**HINT**: You can convert a timestamp type to seconds using the [cast](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.cast.html?highlight=cast) function and the `long` type. Of course, that's just seconds.

In [0]:
from pyspark.sql.functions import round

def calculate_total_time_hours(input_df: DataFrame) -> DataFrame:
### Put your code here.
    return input_df.withColumn("total_time", round((input_df.stop_timestamp.cast("long")- input_df.start_timestamp.cast("long")) / 3600, 2))
    return input_df

result = df.transform(return_stop_transaction_requests). \
    transform(convert_stop_transaction_request_json). \
    transform(
        join_stop_with_start, 
        df.filter(
            (df.action == "StartTransaction") & (df.message_type == 3)). \
            transform(convert_start_transaction_response_json).\
            transform(
                join_with_start_transaction_request, 
                df.filter(
                    (df.action == "StartTransaction") & (df.message_type == 2)
                ).transform(convert_start_transaction_request_json))). \
    transform(convert_start_stop_timestamp_to_timestamp_type). \
    transform(calculate_total_time_hours)

display(result)

#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_calculate_total_time_hours_unit

test_calculate_total_time_hours_unit(spark, calculate_total_time_hours)

#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_calculate_total_time_hours_e2e
    
test_calculate_total_time_hours_e2e(
    df.transform(return_stop_transaction_requests). \
        transform(convert_stop_transaction_request_json). \
        transform(
            join_stop_with_start, 
            df.filter(
                (df.action == "StartTransaction") & (df.message_type == 3)). \
                transform(convert_start_transaction_response_json).\
                transform(
                    join_with_start_transaction_request, 
                    df.filter(
                        (df.action == "StartTransaction") & (df.message_type == 2)
                    ).transform(convert_start_transaction_request_json))). \
        transform(convert_start_stop_timestamp_to_timestamp_type). \
        transform(calculate_total_time_hours),
        display_f=display
)

### EXERCISE: Calculate total_energy and total_time
Let's add a column for the `total_energy`. Subtract `meter_start` from `meter_stop`.

Current schema:

<br/>

```
root
 |-- charge_point_id: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- meter_start: integer (nullable = true)
 |-- meter_stop: integer (nullable = true)
 |-- start_timestamp: timestamp (nullable = true)
 |-- stop_timestamp: timestamp (nullable = true)
 |-- total_time: double (nullable = true) 
```

Expected schema:

<br/>

```
root
 |-- charge_point_id: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- meter_start: integer (nullable = true)
 |-- meter_stop: integer (nullable = true)
 |-- start_timestamp: timestamp (nullable = true)
 |-- stop_timestamp: timestamp (nullable = true)
 |-- total_time: double (nullable = true) 
 |-- total_energy: double (nullable = true) 
```

In [0]:
from pyspark.sql.functions import round

def calculate_total_energy(input_df: DataFrame) -> DataFrame:
### Put your code here.
    return input_df.withColumn("total_energy", 
                               (input_df.meter_stop-input_df.meter_start).cast("double"))
    return input_df

result = df.transform(return_stop_transaction_requests). \
    transform(convert_stop_transaction_request_json). \
    transform(
        join_stop_with_start, 
        df.filter(
            (df.action == "StartTransaction") & (df.message_type == 3)). \
            transform(convert_start_transaction_response_json).\
            transform(
                join_with_start_transaction_request, 
                df.filter(
                    (df.action == "StartTransaction") & (df.message_type == 2)
                ).transform(convert_start_transaction_request_json))). \
    transform(convert_start_stop_timestamp_to_timestamp_type). \
    transform(calculate_total_time_hours). \
    transform(calculate_total_energy)

display(result)

#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_calculate_total_energy_unit

test_calculate_total_energy_unit(spark, calculate_total_energy)

#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.final_charge_time_charge_dispensed_completed_charges import test_calculate_total_energy_e2e
    
test_calculate_total_energy_e2e(
    df.transform(return_stop_transaction_requests). \
        transform(convert_stop_transaction_request_json). \
        transform(
            join_stop_with_start, 
            df.filter(
                (df.action == "StartTransaction") & (df.message_type == 3)). \
                transform(convert_start_transaction_response_json).\
                transform(
                    join_with_start_transaction_request, 
                    df.filter(
                        (df.action == "StartTransaction") & (df.message_type == 2)
                    ).transform(convert_start_transaction_request_json))). \
        transform(convert_start_stop_timestamp_to_timestamp_type). \
        transform(calculate_total_time_hours).\
        transform(calculate_total_energy),
    display_f=display
)

## Wrap-Up & Clean-Up

Congratulations, you have now reached our goal of calculating the total time for which the vehicle was charging and the total charge dispensed by the charge point.

Let's do the clean-up before moving to the next module.

In [0]:
helpers.clean_working_directory()

| Next Module                                                                                                                                  |
|----------------------------------------------------------------------------------------------------------------------------------------------|
| <a href="$../Module 05 - Introducing the Multi-Hop Architecture/5.0 Table of Contents" target="_self">Introducing the Multi-Hop Architecture</a>  | 


%md
&copy; 2024 Thoughtworks. All rights reserved.<br/>